#   Explorando a Arrecadação de Impostos no Brasil ($\text{2000-2026}$)

##  Dependências e Configuração
-   `numpy`, `pandas`: tratamento de dados numéricos, conversões, utilização de `pandas.DataFrame` enquanto estrutura de dados base para o trabalho;
-   `chardet`: identificação *on-line* de *charset encodings* para lidar com dados não tão limpos;
-   `altair`, `calendar`: visualizações; interface temporal

In [67]:
import numpy    as np;
import pandas   as pd;
import altair   as alt;
import scipy    as sci;
import chardet
import calendar

#   dados
DATA: str = r"data/arrecadacao-estado.csv"

#   tema global para altair
alt.theme.enable('fivethirtyeight') 


ThemeRegistry.enable('fivethirtyeight')

## $\S.1 \,$ Leitura dos Dados

In [68]:
#   detectar o character set utilizado nos dados
with open(DATA, "rb") as f:
    raw_data = f.read(10000);
    result = chardet.detect(raw_data);
    encoding = result['encoding'];
    print(f"charset encoding: {encoding}");

#   1.  Leitura e extração dos dados
#   ler com o encoding detectado
df = pd.read_csv(r"data/arrecadacao-estado.csv", encoding=encoding, sep=";");
print(df.head(3));

charset encoding: ISO-8859-1
    Ano      Mês  UF IMPOSTO SOBRE IMPORTAÇÃO IMPOSTO SOBRE EXPORTAÇÃO  \
0  2000  Janeiro  AC                      231                        0   
1  2000  Janeiro  AL                   475088                    33873   
2  2000  Janeiro  AM                 11679405                        0   

  IPI - FUMO IPI - BEBIDAS IPI - AUTOMÓVEIS IPI - VINCULADO À IMPORTACAO  \
0     292096             0                0                          167   
1    1329338        812470                0                       141735   
2    1507146       1791471            27796                      4414483   

  IPI - OUTROS  ... REFIS PAES RETENÇÃO NA FONTE - LEI 10.833, Art. 30  \
0         1558  ...   NaN  NaN                                     NaN   
1      3676847  ...   NaN  NaN                                     NaN   
2      1800346  ...   NaN  NaN                                     NaN   

  PAGAMENTO UNIFICADO OUTRAS RECEITAS ADMINISTRADAS DEMAIS RECEITAS  \
0

## $\S2. \,$ Pré-Processamento
### $\S2.1 \,$ Limpeza dos Dados
Transformação de *strings* de moeda ($\text{R\$}$) para valores numéricos

In [69]:
def clean_currency(col):
    # Se a coluna já for numérica, retorna sem alteração
    if pd.api.types.is_numeric_dtype(col):
        return col

    # Converte para string e remove espaços
    col = col.astype(str).str.strip()
    
    # Remove qualquer caractere que não seja dígito, vírgula ou ponto
    col = col.str.replace(r'[^\d,.]', '', regex=True)
    
    # Substitui vírgula (separador decimal) por ponto
    col = col.str.replace(',', '.', regex=False)
    
    # Função para converter cada valor, tratando pontos de milhar
    def convert_value(x):
        if x == '' or pd.isna(x):
            return np.nan
        # Separa por pontos
        parts = x.split('.')
        if len(parts) == 1:
            # Não tem ponto decimal (ex: '47953915')
            return float(parts[0])
        else:
            # Último segmento é a parte decimal
            decimal = parts[-1]
            # Junta os segmentos anteriores (removendo pontos de milhar)
            integer_part = ''.join(parts[:-1])
            # Reconstrói o número com ponto decimal
            return float(integer_part + '.' + decimal)
    
    return col.apply(convert_value)

#   Converte as strings que representam R$ para float
cols_to_clean       = [c for c in df.columns if c not in ['Ano', 'Mês', 'UF']];
df[cols_to_clean]   = df[cols_to_clean].apply(clean_currency);

### $\S2.2. \,$ Criação de Novas Colunas

In [70]:
#   criar associações mês -> número, número -> mês
meses_pt = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
            'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']

month_to_number = {mes: str(i+1).zfill(2) for i, mes in enumerate(meses_pt)}

#   criação de novas colunas
df['Mês_Num'] = df['Mês'].map(month_to_number)  # cria coluna '01' a '12'
df['Data'] = pd.to_datetime(df['Ano'].astype(str) + '-' + df['Mês_Num'] + '-01')

### $\S2.3. \,$ Transformação dos Dados
Adequação dos dados ao formato `melt` necessário à biblioteca `altair`.

In [71]:
#   colunas identificadoras
id_vars = ['Ano', 'Mês', 'UF', 'Mês_Num', 'Data']

#   colunas valor
value_vars = [c for c in df.columns if c not in id_vars]

#   -> melt
df_melt = df.melt(id_vars=id_vars, value_vars=value_vars, var_name='Imposto', value_name='Valor')
df_melt = df_melt.dropna(subset=['Valor']).query('Valor > 0')  # Remove zeros/NaNs

print(df_melt.shape)

(215309, 7)


## $\S 3. \,$ Visualizações
### $\S 3.1. \,$ Série Temporal: Evolução da Arrecação Total

In [72]:
import altair as alt

# 1. Eixo Y
y_axis = alt.Y(
    'Valor:Q',
    title='Arrecadação Federal Total (R$ bilhões)',
    axis=alt.Axis(
        format='.2s',
        titleFontSize=14,
        labelFontSize=12,
        grid=True,
        titlePadding=20
    )
)

# 2. Eixo X
x_axis = alt.X(
    'Data:T',
    title='Período (Ano)',
    axis=alt.Axis(
        titleFontSize=14,
        labelFontSize=12,
        format='%Y-%m'
    )
)

# 3. Aplicar no gráfico
base = alt.Chart(trend).encode(
    x=x_axis,
    y=y_axis
)

# Gráficos individuais
orig = base.mark_line(color='lightgray', opacity=0.3).encode(y='Valor:Q')
band = base.mark_area(opacity=0.15, color='steelblue').encode(
    y='Min_12:Q',
    y2='Max_12:Q'
)

#   médias móveis, regressão linear
ma = base.mark_line(color='steelblue', strokeWidth=2).encode(y='MA_12:Q')
reg = base.mark_line(color='orange', strokeWidth=2).encode(y='Reg_linear:Q')

chart = (band + ma + reg + orig).properties(
    title=alt.TitleParams(
        text='Evolução da Arrecadação Federal Total (2000-2024)',
        fontSize=18,
        subtitle='Banda de 12 meses (mín/máx), Média Móvel e Tendência Linear'
    ),
    width=700,
    height=400
).configure_axis(
    labelFontSize=12,
    titleFontSize=14,
    grid=True
).configure_legend(
    titleFontSize=14,
    labelFontSize=12
).interactive()

chart.show()
chart.save("visualizations/temporal-series--evolucao-arrecadacao-federal.png")

alt.LayerChart(...)

### $\S 3.2. \,$ Stacked Area: participação relativa dos tributos
O que aconteceu em 2004?

In [73]:
top_impostos = [
    'IRPF', 
    'IRPJ - DEMAIS EMPRESAS', 
    'COFINS - DEMAIS', 
    'CONTRIBUIÇÃO PARA O PIS/PASEP - DEMAIS', 
    'CSLL - DEMAIS',
    'IPI - AUTOMÓVEIS', 
    'IMPOSTO SOBRE IMPORTAÇÃO'
]

df_plot = df_melt[df_melt['Imposto'].isin(top_impostos)]
df_agg = df_plot.groupby(['Data', 'Imposto'])['Valor'].sum().reset_index()

#   color encoding
color_scale = alt.Scale(scheme='tableau10')
color_encoding = alt.Color(
    'Imposto:N',
    scale=color_scale,
    sort=alt.EncodingSortField(field='Valor', op='mean', order='descending'),
    legend=alt.Legend(
        orient='right',
        columns=1,
        title='Tributo',
        titleFontSize=14,
        labelFontSize=12,
        labelLimit=250,
        offset=10
    )
)

#   staked area
area_chart = alt.Chart(df_agg).mark_area(
    opacity=0.85,
    line={'strokeWidth': 1, 'color': 'white'}
).encode(
    x=alt.X(
        'Data:T',
        title='Período (Ano)',
        axis=alt.Axis(format='%Y', titleFontSize=14, labelFontSize=12, grid=False)
    ),
    y=alt.Y(
        'Valor:Q',
        title='Participação Relativa (%)',
        stack='normalize',
        axis=alt.Axis(format='.0%', titleFontSize=14, labelFontSize=12, grid=True)
    ),
    color=color_encoding,
    tooltip=[
        alt.Tooltip('Data:T', title='Data'),
        alt.Tooltip('Imposto:N', title='Tributo'),
        alt.Tooltip('Valor:Q', title='Arrecadação (R$)', format='.2s')
    ]
).properties(
    title=alt.TitleParams(
        text='Tributo COFINS: Impacto da Lei 10.833/2003',
        subtitle=(
            'Elevação de COFINS de coadjuvante para a principal fonte de arrecadação federal'
        ),
        fontSize=18,
        subtitleFontSize=14,
        subtitleColor='#4a4a4a',
        anchor='start',
        offset=10
    ),
    width=700,
    height=450
).interactive()

# marcação de evento histórico
linha_2004 = alt.Chart(pd.DataFrame({'Data': ['2004-01-01']})).mark_rule(
    color='#d62728',
    strokeWidth=2,
    strokeDash=[6, 4]
).encode(
    x='Data:T'
)

# anotação para evento histórico
anotacao_2004 = alt.Chart(pd.DataFrame({
    'Data': ['2004-03-01'],
    'Valor': [0.85],
    'Texto': ['Lei 10.833/2003\n   Regime Não Cumulativo']
})).mark_text(
    align='left',
    baseline='middle',
    dx=15,
    fontSize=13,
    fontWeight='bold',
    color='goldenrod',
    lineBreak='\n'
).encode(
    x='Data:T',
    y=alt.Y('Valor:Q', title='', axis=alt.Axis(labels=False, ticks=False)),
    text='Texto:N'
)

chart = (area_chart + linha_2004 + anotacao_2004).configure_axis(
    gridColor='#e0e0e0',
    labelFontSize=12,
    titleFontSize=14
).configure_legend(
    titleFontSize=14,
    labelFontSize=12,
    labelLimit=250
)

chart.show()
chart.save("visualizations/stacked-area--participacao-relativa-dos-tributos.png")

alt.LayerChart(...)

### $\S 3.3. \,$ Mapa de Calor: Sazonalidade por Estado

In [74]:
# Agregar por UF e Mês (média da arrecadação ao longo de todos os anos)
heat_data = df_melt.groupby(['UF', 'Mês_Num'])['Valor'].mean().reset_index()

# Criar coluna com nome do mês para tooltip
month_labels = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
heat_data['Mês_Nome'] = heat_data['Mês_Num'].map(lambda x: month_labels[int(x)-1])

# Ordenação correta dos meses
month_order = ['01','02','03','04','05','06','07','08','09','10','11','12']

# Heatmap
heatmap = alt.Chart(heat_data).mark_rect(
    stroke='white',      # Borda branca entre as células para melhor distinção
    strokeWidth=0.5
).encode(
    # Eixo X (Meses)
    x=alt.X(
        'Mês_Num:O', 
        title='Período (Mês)',
        sort=month_order,
        axis=alt.Axis(
            labelExpr="['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez'][datum.value - 1]",
            titleFontSize=14,
            labelFontSize=12,
            grid=False
        )
    ),
    
    # Eixo Y (Estados)
    y=alt.Y(
        'UF:N', 
        title='Unidade da Federação',
        axis=alt.Axis(
            titleFontSize=14,
            labelFontSize=11,
            grid=False
        )
    ),
    
    # Cores amigáveis
    color=alt.Color(
        'Valor:Q',
        scale=alt.Scale(
            scheme='viridis',      # Paleta sequencial perceptualmente uniforme
            reverse=False          # Mais claro = maior, mais escuro = menor
        ),
        legend=alt.Legend(
            title='Arrecadação\nMédia (R$)',
            format='.2s',
            titleFontSize=13,
            labelFontSize=11,
            orient='right',
            gradientLength=300
        )
    ),
    
    # Tooltip
    tooltip=[
        alt.Tooltip('UF:N', title='Estado'),
        alt.Tooltip('Mês_Nome:N', title='Mês'),   # Nome do mês
        alt.Tooltip('Valor:Q', title='Média (R$)', format='$.2s')
    ]
).properties(
    title=alt.TitleParams(
        text='Padrões Sazonais da Arrecadação Federal por Estado',
        subtitle='Média mensal histórica (2000 - 2026)',
        fontSize=17,
        subtitleFontSize=13,
        subtitleColor='#555555',
        anchor='start'
    ),
    width=alt.Step(32),   # Largura fixa por coluna (mês)
    height=alt.Step(18)   # Altura fixa por linha (estado)
).configure_view(
    strokeWidth=0         # Remove a borda externa do gráfico
).interactive()

heatmap.show()
heatmap.save("visualizations/heatmap--sazonalidade-arrecadacao-por-estado.png")

alt.Chart(...)